In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

In [3]:
# summarising useful metrics for exploring dataframes

def df_summary(df):
    return pd.DataFrame({
        'dtype': df.dtypes,
        'null_count': df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
        'unique_values': df.nunique(),
        'min': df.min(numeric_only=False),
        'max': df.max(numeric_only=False),
    })


# Data: Sources, Structure and Preparation

This notebook is a technical appendix to the main series. It does not contain analysis. Its job is to document where the data comes from, what it looks like before any cleaning, what decisions were made during cleaning, and what shape it is in when it reaches the analysis notebooks.

Readers who want to understand or reproduce the analysis will find everything they need here. Readers who are primarily interested in the statistical results can skip this notebook and join the series at Article 1.

## 1. Data Sources

The data comes from two separate sources, covering different competitions at different levels of granularity.

**National league data** was scraped from football-data.co.uk, which publishes historical match results for the top European divisions. The dataset covers five leagues across fifteen seasons, from 2010/11 through 2024/25:

* Serie A (Italy)
* Premier League (England)
* La Liga (Spain)
* Bundesliga (Germany)
* Ligue 1 (France)

Each row in the national league data represents one match, with separate columns for the home team and the away team. The columns relevant to this series are fouls committed, yellow cards and red cards, recorded separately for each side.

**International competition data** was scraped from FBref and supplemented from ESPN for seasons where FBref had gaps. This covers the UEFA Champions League, Europa League and Europa Conference League. Unlike the national data, international data is available only at season level: each row represents one team's aggregate statistics across an entire European campaign. The implications of this granularity difference are discussed at the end of this notebook.


In [4]:
with open('../data/raw/european_leagues_data.pkl', 'rb') as f:
    leagues_data = pickle.load(f)

with open('../data/raw/european_comps_data.pkl', 'rb') as f:
    european_comps_data = pickle.load(f)

print("National leagues:", list(leagues_data.keys()))
print("International competitions:", list(european_comps_data.keys()))

National leagues: ['Serie_A', 'Premier_League', 'La_Liga', 'Bundesliga', 'Ligue_1']
International competitions: ['UCL', 'UEL', 'UECL']


## 2. National League Data: Structure

Before cleaning anything, it is worth understanding what the raw data looks like. Each league is stored as a separate dataframe with the same column structure.

In [5]:
df_sample = leagues_data['Serie_A']
print(f"Shape: {df_sample.shape}")
print()
print(df_sample.dtypes)
print()
df_sample.head(3)

Shape: (5625, 22)

Date            str
HomeTeam        str
AwayTeam        str
FTHG        float64
FTAG        float64
FTR             str
HTHG        float64
HTAG        float64
HTR             str
HF          float64
AF          float64
HY          float64
AY          float64
HR          float64
AR          float64
HS          float64
AS          float64
HST         float64
AST         float64
HC          float64
AC          float64
season          str
dtype: object



,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HF,...,AY,HR,AR,HS,AS,HST,AST,HC,AC,season
0,23/08/2025,Genoa,Lecce,0.0,0.0,D,0.0,0.0,D,16.0,...,3.0,0.0,0.0,5.0,7.0,2.0,0.0,3.0,7.0,2526
1,23/08/2025,Sassuolo,Napoli,0.0,2.0,A,0.0,1.0,A,17.0,...,1.0,1.0,0.0,7.0,13.0,2.0,4.0,1.0,2.0,2526
2,23/08/2025,Milan,Cremonese,1.0,2.0,A,1.0,1.0,D,10.0,...,4.0,0.0,0.0,24.0,4.0,6.0,3.0,9.0,2.0,2526


## National Leagues Data Dictionary

- `Date`: `str` - Match Date (dd/mm/yy)
- `HomeTeam`: `str` - Home Team
- `AwayTeam`: `str` - Away Team
- `FTHG` - `float64` - Results - Full Time Home Team Goals
- `FTAG` - `float64`- Results - Full Time Away Team Goals
- `FTR` - `str` - Results - Full Time Result (H=Home Win, D=Draw, A=Away Win)
- `HTHG` - `float64` - Results - Half Time Home Team Goals
- `HTAG` - `float64` - Results - Half Time Away Team Goals
- `HTR` - `str` - Results - Half Time Result (H=Home Win, D=Draw, A=Away Win)
- `HF` - `float64` - Discipline - Home Team Fouls Committed
- `AF` - `float64` - Discipline - Away Team Fouls Committed
- `HY` - `float64` - Discipline - Home Team Yellow Cards
- `AY` - `float64` - Discipline - Away Team Yellow Cards
- `HR` - `float64` - Discipline - Home Team Red Cards
- `AR` - `float64` - Discipline - Away Team Red Cards
- `HS` - `float64` - Attack/Defence - Home Team Shots
- `AS` - `float64` - Attack/Defence - Away Team Shots
- `HST` - `float64` - Attack/Defence - Home Team Shots on Target
- `AST` - `float64` - Attack/Defence - Away Team Shots on Target
- `HC` - `float64` - Attack/Defence - Home Team Corners
- `AC` - `float64` - Attack/Defence - Away Team Corners
- `season` - `str` - Season (yyyy)

The columns relevant to this project are:

- `HF` / `AF`: fouls committed by the home and away team
- `HY` / `AY`: yellow cards received by the home and away team
- `HR` / `AR`: red cards received by the home and away team
- `HomeTeam` / `AwayTeam`: team names
- `Date`: match date
- `season`: season identifier in the format `YYZZ`, for example `2425` for the 2024/25 season

The remaining columns (shots, corners, goals, half-time results) are available and used in the analysis where relevant.

One thing to note immediately: all numeric columns are stored as `float64` rather than integers. This is because pandas promotes integer columns to float when any null values are present. We will handle nulls in the next section.

## 3. Cleaning: Missing Values and Duplicates

For each league we check two things: missing values in the discipline columns, and duplicate rows. These are the only issues that would affect the analysis directly.

In [6]:
discipline_cols = ['HF', 'AF', 'HY', 'AY', 'HR', 'AR']

cleaning_report = []

for league_name, df in leagues_data.items():
    n_raw = len(df)
    n_nulls = df[discipline_cols].isna().any(axis=1).sum()
    n_dupes = df.duplicated().sum()
    cleaning_report.append({
        'league': league_name,
        'rows_raw': n_raw,
        'null_discipline_rows': n_nulls,
        'duplicate_rows': n_dupes,
        'rows_after_cleaning': n_raw - n_nulls - n_dupes
    })

pd.DataFrame(cleaning_report).set_index('league')


,rows_raw,null_discipline_rows,duplicate_rows,rows_after_cleaning
league,,,,
Serie_A,5625,7,2,5616
Premier_League,5630,1,0,5629
La_Liga,5610,0,0,5610
Bundesliga,4527,1,0,4526
Ligue_1,5315,5,0,5310


The losses are small: a handful of matches spread across multiple seasons and leagues where the discipline data was not recorded. In each case we verified that the missing rows correspond to known collection gaps at the source, not to errors introduced during scraping. Dropping them loses less than 0.2% of the data in the worst affected season and has no meaningful effect on any league-level statistic.

In [7]:
cleaned = {}

for league_name, df in leagues_data.items():
    cleaned[league_name] = df.dropna(subset=discipline_cols).drop_duplicates().copy()
    n_before = len(leagues_data[league_name])
    n_after = len(cleaned[league_name])
    print(f"{league_name:20s}  {n_before:5d} rows  ->  {n_after:5d} rows  ({n_before - n_after} dropped)")


Serie_A                5625 rows  ->   5618 rows  (7 dropped)
Premier_League         5630 rows  ->   5629 rows  (1 dropped)
La_Liga                5610 rows  ->   5610 rows  (0 dropped)
Bundesliga             4527 rows  ->   4526 rows  (1 dropped)
Ligue_1                5315 rows  ->   5310 rows  (5 dropped)


One small note on Serie A: the cleaning report suggested 9 rows to drop (7 nulls and 2 duplicates) but only 7 were actually removed. The two duplicate rows were a subset of the null rows, so dropping nulls first eliminated them before the duplicate check ran. This is a common subtlety in data cleaning: counts computed independently do not always add up when the same rows qualify under multiple criteria.

## 4. Reshaping to Team-Match Format

The raw data records each match once, with separate columns for the home and away team. This is convenient for match-level questions but awkward for team-level questions: to ask how many fouls a specific team committed across a season, we would need to check two different columns depending on whether they were playing at home or away.

We reshape the data so that each match generates two rows, one from the home team's perspective and one from the away team's perspective. Every column becomes team-centric: `fouls_committed` always refers to the team in that row, and `fouls_received` always refers to what the opponent committed against them. This makes all team-level aggregations straightforward and symmetric.

In [10]:
def reshape_to_team_matches(df, league_name):
    home = df[[
        'season', 'Date', 'HomeTeam', 'AwayTeam',
        'HY', 'HR', 'HF', 'AF',
        'HS', 'AS', 'HST', 'AST',
        'HC', 'AC', 'FTHG', 'FTAG', 'FTR'
    ]].copy()

    home.columns = [
        'season', 'date', 'team', 'opponent',
        'yellow_cards', 'red_cards', 'fouls_committed', 'fouls_received',
        'shots', 'shots_opponent', 'shots_on_target', 'shots_on_target_opponent',
        'corners', 'corners_opponent', 'goals_scored', 'goals_conceded', 'match_result'
    ]
    home['venue'] = 'home'
    home['team_result'] = home['match_result'].map({'H': 'W', 'D': 'D', 'A': 'L'})

    away = df[[
        'season', 'Date', 'AwayTeam', 'HomeTeam',
        'AY', 'AR', 'AF', 'HF',
        'AS', 'HS', 'AST', 'HST',
        'AC', 'HC', 'FTAG', 'FTHG', 'FTR'
    ]].copy()

    away.columns = [
        'season', 'date', 'team', 'opponent',
        'yellow_cards', 'red_cards', 'fouls_committed', 'fouls_received',
        'shots', 'shots_opponent', 'shots_on_target', 'shots_on_target_opponent',
        'corners', 'corners_opponent', 'goals_scored', 'goals_conceded', 'match_result'
    ]
    away['venue'] = 'away'
    away['team_result'] = away['match_result'].map({'H': 'L', 'D': 'D', 'A': 'W'})

    combined = pd.concat([home, away], ignore_index=True)
    combined['league'] = league_name
    combined['date'] = pd.to_datetime(combined['date'], dayfirst=True, format='mixed')

    return combined.sort_values(['season', 'date', 'team']).reset_index(drop=True)


In [11]:
team_matches = pd.concat(
    [reshape_to_team_matches(df, league_name) for league_name, df in cleaned.items()],
    ignore_index=True
)

print(f"Total rows:  {len(team_matches):,}")
print(f"Leagues:     {team_matches['league'].nunique()}")
print(f"Seasons:     {team_matches['season'].nunique()}")
print(f"Date range:  {team_matches['date'].min().date()} to {team_matches['date'].max().date()}")
print()
team_matches.head(4)


Total rows:  53,386
Leagues:     5
Seasons:     15
Date range:  2011-08-05 to 2026-03-22



,season,date,team,opponent,yellow_cards,red_cards,fouls_committed,fouls_received,shots,shots_opponent,shots_on_target,shots_on_target_opponent,corners,corners_opponent,goals_scored,goals_conceded,match_result,venue,team_result,league
0,1112,2011-09-09,Lazio,Milan,2.0,0.0,15.0,16.0,12.0,18.0,5.0,8.0,1.0,3.0,2.0,2.0,D,away,D,Serie_A
1,1112,2011-09-09,Milan,Lazio,2.0,0.0,16.0,15.0,18.0,12.0,8.0,5.0,3.0,1.0,2.0,2.0,D,home,D,Serie_A
2,1112,2011-09-10,Cesena,Napoli,2.0,1.0,14.0,12.0,11.0,18.0,3.0,6.0,4.0,6.0,1.0,3.0,A,home,L,Serie_A
3,1112,2011-09-10,Napoli,Cesena,3.0,0.0,12.0,14.0,18.0,11.0,6.0,3.0,6.0,4.0,3.0,1.0,A,away,W,Serie_A


Each match now appears twice, once for each team. The columns `fouls_committed` and `fouls_received` are always from the perspective of the team in that row, regardless of venue. The `venue` column records whether that team was playing at home or away, and `team_result` records the outcome from their perspective.

A quick sanity check: row 0 shows Lazio away at Milan, with `fouls_committed = 15` and `fouls_received = 16`. Row 1 shows Milan at home to Lazio, with `fouls_committed = 16` and `fouls_received = 15`. The numbers are mirrored correctly.


## 5. Season Completeness Check

Cleaning rows is necessary but not sufficient. A dataset can pass a null check and still be missing an entire round of fixtures. Before committing to any season as a reference point for analysis, we need to verify that the expected number of matches is present.

The expected count follows directly from the league format. In a home-and-away competition with N teams, every team plays every other team twice, giving N × (N − 1) matches per season. For a 20-team league that is 380 matches; for an 18-team league it is 306.

One format change matters here: Ligue 1 reduced from 20 to 18 teams starting with the 2023/24 season, dropping the expected match count from 380 to 306. Everything else remained constant across the fifteen seasons in the dataset.

In [12]:
def expected_match_count(league, season):
    if league == 'Bundesliga':
        return 306
    if league == 'Ligue_1':
        return 306 if season >= '2324' else 380
    return 380

records = []

for league_name, df in cleaned.items():
    counts = df.groupby('season').size().reset_index(name='actual')
    counts['league'] = league_name
    counts['expected'] = counts.apply(
        lambda r: expected_match_count(league_name, r['season']), axis=1
    )
    counts['missing'] = counts['expected'] - counts['actual']
    counts['complete'] = counts['missing'] == 0
    records.append(counts)

completeness = (
    pd.concat(records, ignore_index=True)
    .sort_values(['league', 'season'])
    [['league', 'season', 'expected', 'actual', 'missing', 'complete']]
    .reset_index(drop=True)
)

completeness


,league,season,expected,actual,missing,complete
0,Bundesliga,1112,306,306,0,True
1,Bundesliga,1213,306,306,0,True
2,Bundesliga,1314,306,306,0,True
3,Bundesliga,1415,306,306,0,True
4,Bundesliga,1516,306,306,0,True
...,...,...,...,...,...,...
70,Serie_A,2122,380,380,0,True
71,Serie_A,2223,380,380,0,True
72,Serie_A,2324,380,380,0,True
73,Serie_A,2425,380,380,0,True


In [13]:
incomplete = completeness[~completeness['complete']]

if incomplete.empty:
    print("All seasons are complete across all five leagues.")
else:
    print(f"{len(incomplete)} season(s) with missing matches:\n")
    print(incomplete.to_string(index=False))


11 season(s) with missing matches:

        league season  expected  actual  missing  complete
    Bundesliga   2425       306     305        1     False
    Bundesliga   2526       306     243       63     False
       La_Liga   2526       380     290       90     False
       Ligue_1   1112       380     378        2     False
       Ligue_1   1617       380     379        1     False
       Ligue_1   1920       380     279      101     False
       Ligue_1   2526       306     242       64     False
Premier_League   2526       380     309       71     False
       Serie_A   1213       380     379        1     False
       Serie_A   1415       380     379        1     False
       Serie_A   2526       380     300       80     False


In [14]:
def classify_season(league, season, missing):
    if season == '2526':
        return 'current_season'
    if league == 'Ligue_1' and season == '1920':
        return 'covid_truncated'
    if missing <= 2:
        return 'complete'
    return 'incomplete'

completeness['status'] = completeness.apply(
    lambda r: classify_season(r['league'], r['season'], r['missing']), axis=1
)

print(completeness['status'].value_counts())
print()
completeness[completeness['status'] != 'complete']


status
complete           69
current_season      5
covid_truncated     1
Name: count, dtype: int64



,league,season,expected,actual,missing,complete,status
14,Bundesliga,2526,306,243,63,False,current_season
29,La_Liga,2526,380,290,90,False,current_season
38,Ligue_1,1920,380,279,101,False,covid_truncated
44,Ligue_1,2526,306,242,64,False,current_season
59,Premier_League,2526,380,309,71,False,current_season
74,Serie_A,2526,380,300,80,False,current_season


The incomplete seasons fall into three categories.

**Current season (2025/26)**: all five leagues are still in progress at the time of writing. These are excluded from all analysis in the main series, which uses complete seasons only.

**COVID-truncated season (Ligue 1, 2019/20)**: the French top flight [cancelled the remainder of the 2019/20 season](https://en.wikipedia.org/wiki/2019%E2%80%9320_Ligue_1) due to the COVID-19 outbreak, leaving 279 of 380 matches played. Season-level totals and averages for this year are not comparable to a full season and it is excluded from any analysis that relies on season aggregates.

**Minor gaps (1 to 2 matches missing)**: five season-league combinations are missing one or two matches. At less than 0.6% of a full season these gaps are negligible for any league-level statistic and these seasons are retained and treated as complete throughout the analysis.

In [15]:
excluded_seasons = completeness[
    completeness['status'].isin(['current_season', 'covid_truncated'])
][['league', 'season', 'status']]

print("Seasons excluded from the main analysis:\n")
print(excluded_seasons.to_string(index=False))


Seasons excluded from the main analysis:

        league season          status
    Bundesliga   2526  current_season
       La_Liga   2526  current_season
       Ligue_1   1920 covid_truncated
       Ligue_1   2526  current_season
Premier_League   2526  current_season
       Serie_A   2526  current_season


In [16]:
excluded = set(zip(excluded_seasons['league'], excluded_seasons['season']))

team_matches_clean = team_matches[
    ~team_matches.apply(lambda r: (r['league'], r['season']) in excluded, axis=1)
].copy()

print(f"Full dataset:     {len(team_matches):,} rows")
print(f"After exclusions: {len(team_matches_clean):,} rows")
print(f"Rows removed:     {len(team_matches) - len(team_matches_clean):,}")


Full dataset:     53,386 rows
After exclusions: 50,060 rows
Rows removed:     3,326


In [17]:
with open('../data/processed/team_matches.pkl', 'wb') as f:
    pickle.dump(team_matches_clean, f)

print("Saved: data/processed/team_matches.pkl")
print(f"Leagues: {sorted(team_matches_clean['league'].unique())}")
print(f"Seasons: {sorted(team_matches_clean['season'].unique())}")


Saved: data/processed/team_matches.pkl
Leagues: ['Bundesliga', 'La_Liga', 'Ligue_1', 'Premier_League', 'Serie_A']
Seasons: ['1112', '1213', '1314', '1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425']


The cleaned dataset covers 14 complete seasons across five leagues, from 2011/12 through 2024/25. The 2025/26 current season and the COVID-truncated Ligue 1 2019/20 season are excluded. All downstream analysis notebooks load from this file.

## 6. A Note on the International Data

The international competition data requires separate treatment and is not fully processed here. The reason is a structural difference that matters for any comparison: national league data is at match level, while international data is at season level.

When a team's season is summarised as a single row of totals, the uncertainty around that total is very different from the uncertainty around individual match observations. A team that received 40 yellow cards across 13 Champions League matches gives a rate of roughly 3.1 per match, but we cannot see how that rate varied across those 13 games, whether it was driven by one particularly ill-disciplined night or distributed evenly across the campaign. The match-level data from national leagues does not have this problem.

Any analysis that mixes the two sources requires explicit aggregation of the national data to season level first, so that both datasets are on equal footing. That aggregation step, and its consequences for what we can and cannot conclude, are discussed in the article where the international data is first used.


### A note on specific Ligue 1 matches

Two of the dropped rows are worth naming explicitly.

**Bastia vs Lyon (2015/16)**: this match was officially abandoned following an attack on Lyon players by Bastia supporters and was never replayed. There are no statistics to recover because the match was never completed. This is not a data quality issue.

**Lyon vs Marseille and Caen vs Nancy (2011/12)**: both matches are missing fouls data (`HF` and `AF`). Since fouls are the key normalisation variable for the discipline analysis throughout this series, these rows cannot be imputed and are dropped.

In all other cases the missing discipline data reflects collection gaps at the source rather than real-world events.
